In [0]:
RESULTS_CSV = "/Volumes/workspace/drift_monitor/results/score_results.csv"
GOLD_RESULTS = "workspace.drift_monitor.gold_detector_results"

results = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RESULTS_CSV)
)
results.write.mode("overwrite").saveAsTable(GOLD_RESULTS)
display(results)

In [0]:
%sql
WITH graded AS (
    SELECT
        scenario,
        detector,
        CASE
            WHEN scenario = 'no_drift' THEN CAST(false_alarms AS STRING) || ' false alarm(s)'
            WHEN detector = 'psi_type' AND scenario <> 'type_shift_30pct' THEN 'not watched'
            WHEN detector <> 'psi_type' AND scenario = 'type_shift_30pct' THEN 'not watched'
            WHEN missed THEN 'missed'
            ELSE 'caught (' || CAST(CAST(delay_windows AS INT) AS STRING) || ')'
        END AS result
    FROM workspace.drift_monitor.gold_detector_results
)
SELECT *
FROM graded
PIVOT (
    FIRST(result) FOR detector IN ('psi_amount', 'psi_type', 'ks_d_amount', 'adwin_amount')
)
ORDER BY scenario